# CT-CBM 

---
Goal of the notebook: computed the combined score for ranking concepts then put it in cluster wise selection list (in pickle file).

**Inputs of the notebook**:

- df_train_{annotation}.csv
- df_val_{annotation}.csv
- df_test_{annotation}.csv

**!!! PS: le dataset de val n'a pas besoin d'être annoté !!!**

**Output of the notebook**:
- combined score for each concept  **.json**
- concepts to add at each iteration when we based ourselve on combined score and cluster of concepts **.json**

**exemple.** :
Step 1: Concepts ['business developments', 'cultural significance', 'performance coverage', 'internet platforms', 'cultural traditions', 'event preparation', 'audience response', 'visual media', 'career milestones'] → Coverage 94.79%
Step 2: Concepts ['commentary', 'innovation narratives', 'seasonal focus', 'media networks', 'taste experiences', 'gastronomic techniques',   'statistics', 'performance venues', 'artistic development'] → Coverage 99.10%


# Setup & Imports


In [ ]:
!pip install hdbscan
!pip install umap-learn

In [ ]:
# dbutils.library.restartPython()  # Databricks only

In [ ]:
import sys
sys.path.append('../run_experiments/')
sys.path.append('../run_experiments/scripts')
sys.path.append('../run_experiments/models')
sys.path.append('../run_experiments/data')

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import pickle
import json
from tqdm import tqdm

In [ ]:
#code for autoreload script associated with jupyter notebook
%load_ext autoreload
%autoreload 2

# 0. Configuration & Dispatcher Initialization

In [ ]:
from unified_config import load_config
from pipeline_dispatcher import ModalityDispatcher

# 1.SETUP ENVIRONMENT VARIABLES

In [ ]:
dataset = 'PBC'          # 'n24news', 'agnews', 'dbpedia', 'medical', 'ledgar'
annotation = 'C3M'           # 'C3M', 'our_annotation', 'cb_llm'
combine_type = 'None'  # 'concat', 'combine', 'None'
model_name = 'clip'          # 'clip', 'blip', 'bert-base-uncased', 'gemma', etc.
modality_mode = 'image' # 'text'/ 'image'/'multimodal'
agg_mode = 'abs'  # toujours absolute value pour les attributions LIG
agg_scope = 'all' # ON TOUCHE PAS ça
path_to_input = "data/datasets/PBC" # input/output path
path_to_output = "data/datasets/PBC/C3M_annotation/outputs_concept_scoring" # input/output path

# Load configuration
config = load_config(model_name, dataset, annotation, modality_mode, combine_type, path_to_input, path_to_output)
config.annotation = annotation
config.cavs_type = 'mean' if config.annotation != 'cb_llm' else 'regression'
config.agg_mode = agg_mode
config.agg_scope = agg_scope
config.combine_type = combine_type

# ✨ Create dispatcher - handles modality automatically
dispatcher = ModalityDispatcher(config)
print(f"   Modality: {dispatcher.modality_mode}")



# 2. Data Loading

In [ ]:
# load_from_csv

data_tuple = dispatcher.load_data()

# Unpack based on what we got
if len(data_tuple) == 6:
    train_loader, test_loader, val_loader, df_aug_train, df_aug_val, df_aug_test = data_tuple
    print(f" Loaded augmented data")
print(f"\n Data shapes:")
print(f"   Train: {len(train_loader.dataset)} samples")
print(f"   Val: {len(val_loader.dataset)} samples")
print(f"   Test: {len(test_loader.dataset)} samples")


# 1. Clustering of concept

In [ ]:
from clustering import cluster_and_visualize_topics

cluster_and_visualize_topics(
    config=config,
    annotation=annotation,
    min_cluster_size=2,
    random_state=42,
    df_aug_train = df_aug_train
)

# 2. Blackbox model training

Indispensable au calcul des CAVS

In [ ]:
# ✨ Load or train black-box model (automatic routing by model type)
black_box_model, embedder_tokenizer = dispatcher.load_or_train_blackbox(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    force_retrain=False  # Set to True to force retraining
)
print(f" Black-box model ready")

# 3. Compute Cavs vectors

In [ ]:
# Extract concept list from augmented data
concept_name_list = dispatcher.extract_concept_list(df_aug_train)

# ✨ Load or compute CAVs (checks if file exists first)
data_source = train_loader if config.modality_mode in ['image', 'multimodal'] else df_aug_train
cavs = dispatcher.load_or_compute_cavs(
    data_source=data_source,
    baseline_model=black_box_model,
    concept_list=concept_name_list,
    embedder_tokenizer = embedder_tokenizer,
    processor=embedder_tokenizer, # embedder_tokenizer is call processor for blip and clip
    force_recompute=False
)

if cavs:
    print(f" Computed {len(cavs)} CAVs")
    print(f"   Sample concepts: {list(cavs.keys())[:]}...")
else:
    print(f" CAVs will be loaded from file by TCAV ranker")

calculer le TCAVS score à partir des moyennes des embedding

In [ ]:
# ✨ Create TCAV ranker (automatic routing by dataset/model)
TCAVS_ranker = dispatcher.create_tcav(
    concepts=concept_name_list,
    baseline_model=black_box_model,
    embedder_tokenizer=embedder_tokenizer,
    verbose=True
)
print(f" TCAV ranker created")

# Compute TCAV scores
tcav_scores = dispatcher.compute_tcav_scores(
    tcav_ranker=TCAVS_ranker,
    train_loader=train_loader,
    max_examples_per_concept=2
)
print(f" TCAV scores computed for {len(tcav_scores)} classes")

# Rank concepts by TCAV scores
sorted_macro_concepts = dispatcher.rank_and_save_concepts(tcav_scores)
print(f" Ranked and saved {len(sorted_macro_concepts)} macro concepts")

# Display top concepts
print(f"\nTop 10 concepts:", sorted_macro_concepts[:10])

# 4.LIG_ranking

In [ ]:
df_updated, sorted_lig = dispatcher.compute_lig_ranking(
    black_box_model=black_box_model,
    train_loader=train_loader,
    train_df=df_aug_train,
    cavs=cavs,
    mode=config.agg_mode,
    agg_scope=config.agg_scope
)
print(f"Top 10 LIG: {sorted_lig[:10]}")

# 5. computation_of_combined_score

## 5.1.Calculer le concept identifiability score

concept identifiability score = F1 score ou R-squared de detection sur le validation set 

prendre le threshold en se basant sur la mediane cosine similarity sur le train set (pour l'utiliser comme threshold sur le test)


In [ ]:
# 4. F1 Identifiability Score (NOUVELLE MÉTHODE !)
result = dispatcher.compute_identifiability_score(
    data_source=train_loader if config.modality_mode in ['image', 'multimodal'] else df_aug_train,
    model = black_box_model,
    embedder_tokenizer= embedder_tokenizer,
    cavs=cavs,
    f1_cutoff=None
)

## 5.2 Compute the Global indice for ranking concepts

In [ ]:
# 7. SCORES COMBINÉS (NOUVELLE MÉTHODE !)
plot_df = dispatcher.compute_combined_scores(df_aug_train, top_n=10)

# END for computing the combined score

## 6. Score by cluster by TCAVS et BY LIG)

In [ ]:
if config.annotation != "cb_llm":
    results_coverage = dispatcher.coverage_analysis(df_aug_train = df_aug_train, TCAVS_or_LIG = 'both')
else:
    print("pas de mesure de coverage puisque pas d'occurence (valeur différente de 0 et 1) ")